# Iris データセット探索

Kotlin で Iris データセットを探索し、決定木モデルの性能を評価します。

In [ ]:
// プロジェクトのクラスパスを追加
// （IntelliJ IDEA の Kotlin Notebook プラグインを使用する場合は自動的に設定されます）

In [2]:
// ライブラリのインポート
import ml.IrisClassifier
import java.io.File

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[2], line 2, column 8: Unresolved reference: ml

## データの読み込みと概要表示

In [ ]:
val classifier = IrisClassifier()
val (X, y) = classifier.loadData("src/main/resources/data/iris.csv")

println("データ形状: ${X.size} サンプル × ${X[0].size} 特徴量")
println("クラス数: ${y.distinct().size} 種類")
println("\n最初の5サンプル:")
X.take(5).forEachIndexed { i, features ->
    println("サンプル ${i+1}: [${features.joinToString(", ")}] -> ${y[i]}")
}

## 基本統計量の確認

In [ ]:
// 各特徴量の統計情報を計算
val featureNames = listOf("Sepal Length", "Sepal Width", "Petal Length", "Petal Width")

println("\n基本統計量:")
for (featureIdx in 0 until 4) {
    val values = X.map { it[featureIdx] }
    val mean = values.average()
    val std = kotlin.math.sqrt(values.map { (it - mean) * (it - mean) }.average())
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    
    println("${featureNames[featureIdx]}: 平均=${"%%.3f".format(mean)}, " +
           "標準偏差=${"%%.3f".format(std)}, " +
           "最小=${"%%.3f".format(min)}, " +
           "最大=${"%%.3f".format(max)}")
}

## クラスの分布

In [ ]:
println("\n品種の分布:")
val total = y.size.toDouble()
y.distinct().sorted().forEach { species ->
    val count = y.count { it == species }
    val percentage = (count / total * 100)
    println("  $species: $count 件 (${"%%.1f".format(percentage)}%)")
}

## モデル訓練と評価

In [ ]:
println("\n=== モデル訓練 ===")
classifier.train(X, y)

val accuracy = classifier.evaluate(X, y)
println("\nモデル正解率: ${"%%.2f".format(accuracy * 100)}%")

## 混同行列の表示

In [ ]:
val predictions = classifier.predict(X)
val species = y.distinct().sorted()

println("\n混同行列:")
println("実際 \\ 予測 | " + species.joinToString(" | "))
println("-".repeat(60))

species.forEach { actualSpecies ->
    val actualIndices = y.indices.filter { y[it] == actualSpecies }
    print("%-15s | ".format(actualSpecies))
    species.forEach { predSpecies ->
        val count = actualIndices.count { predictions[it] == predSpecies }
        print("%3d | ".format(count))
    }
    println()
}

## 個別予測の例

In [ ]:
println("\n=== 個別予測の例 ===")
val testSamples = arrayOf(
    doubleArrayOf(5.1, 3.5, 1.4, 0.2),  // setosa の特徴
    doubleArrayOf(6.5, 3.0, 5.2, 2.0),  // virginica の特徴
    doubleArrayOf(5.7, 2.8, 4.1, 1.3)   // versicolor の特徴
)

testSamples.forEachIndexed { i, sample ->
    val prediction = classifier.predict(arrayOf(sample))[0]
    println("サンプル ${i+1}: [${sample.joinToString(", ")}] -> 予測: $prediction")
}

## モデルの保存

In [ ]:
val modelPath = "model/iris_model.ser"
classifier.saveModel(modelPath)
println("\nモデルを保存しました: $modelPath")